# Preparing Training Data for OmniSVG

This notebook demonstrates how to prepare training data for the OmniSVG model. We will:

1. Download sample SVG datasets
2. Preprocess and simplify SVGs
3. Generate text descriptions (optional)
4. Convert to the training format
5. Save the processed data

In [ ]:
import os
import sys
import json
import random
import requests
import xml.etree.ElementTree as ET
from tqdm import tqdm
import pandas as pd
from PIL import Image
import io
import matplotlib.pyplot as plt
import numpy as np
import cairosvg

# Add parent directory to path to import OmniSVG modules
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.data_processing import SVGProcessor
from src.utils import set_seed, save_json, visualize_svg

# Set random seed for reproducibility
set_seed(42)

## Download Sample SVG Datasets

We'll use some publicly available SVG datasets, such as:

1. OpenMoji - Open source emojis for designers, developers and everyone else.
2. FIGR-8-SVG - Simple vector graphics dataset.

Let's start by downloading some samples:

In [ ]:
# Create data directories
raw_data_dir = "../data/raw"
processed_data_dir = "../data/processed"

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(processed_data_dir, exist_ok=True)

In [ ]:
# Download FIGR-8-SVG dataset (or subset)
def download_figr8_sample(num_samples=100):
    # Download a subset of FIGR-8-SVG from GitHub
    figr8_dir = os.path.join(raw_data_dir, "figr8")
    os.makedirs(figr8_dir, exist_ok=True)
    
    # Base URL for raw files on GitHub
    base_url = "https://raw.githubusercontent.com/marcdemers/FIGR-8-SVG/master/dataset/figr8.csv"
    
    try:
        response = requests.get(base_url)
        if response.status_code == 200:
            # Save the full CSV file
            with open(os.path.join(figr8_dir, "figr8.csv"), "w") as f:
                f.write(response.text)
            
            # Load data and extract SVGs
            df = pd.read_csv(os.path.join(figr8_dir, "figr8.csv"))
            
            # Take a random sample
            if num_samples and num_samples < len(df):
                df = df.sample(num_samples, random_state=42)
            
            # Extract SVGs and save to individual files
            svg_samples = []
            for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting SVGs"):
                svg_content = row['svg']
                filename = f"figr8_{idx}.svg"
                
                # Save SVG file
                with open(os.path.join(figr8_dir, filename), "w") as f:
                    f.write(svg_content)
                
                # Add to samples list
                svg_samples.append({
                    "file_path": os.path.join(figr8_dir, filename),
                    "category": row.get('category', 'unknown')
                })
            
            print(f"Downloaded {len(svg_samples)} SVG samples from FIGR-8-SVG dataset")
            return svg_samples
        else:
            print(f"Failed to download FIGR-8-SVG dataset: {response.status_code}")
            return []
    except Exception as e:
        print(f"Error downloading FIGR-8-SVG dataset: {e}")
        return []

In [ ]:
# Download or use existing data
figr8_samples = download_figr8_sample(num_samples=100)

## Initialize SVG Processor

Now let's initialize our SVG processor to handle the SVG simplification and tokenization:

In [ ]:
# Initialize SVG processor
svg_processor = SVGProcessor(
    base_tokenizer_name="Qwen/Qwen2.5-VL-3B",
    max_svg_len=8192,
    viewbox_size=200
)

## Process SVGs

Now we will process the SVGs, simplify them, and prepare them for training:

In [ ]:
def process_svg_samples(samples, svg_processor):
    processed_samples = []
    
    for sample in tqdm(samples, desc="Processing SVGs"):
        file_path = sample['file_path']
        category = sample.get('category', 'unknown')
        
        try:
            # Read SVG content
            with open(file_path, 'r', encoding='utf-8') as f:
                svg_content = f.read()
            
            # Simplify SVG
            simplified_svg = svg_processor.simplify_svg(svg_content)
            
            # Convert to commands
            commands = svg_processor.svg_to_commands(simplified_svg)
            
            # Convert back to SVG to ensure consistency
            processed_svg = svg_processor.commands_to_svg(commands)
            
            # Generate a basic description
            description = f"A {category} icon"
            
            processed_samples.append({
                'svg': processed_svg,
                'text': description,
                'category': category,
                'original_file': file_path
            })
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
    
    return processed_samples

In [ ]:
# Process SVG samples
processed_samples = process_svg_samples(figr8_samples, svg_processor)

## Visualize Some Processed SVGs

Let's visualize some of the processed SVGs to make sure they look good:

In [ ]:
def display_svg_samples(samples, num_samples=5):
    if len(samples) == 0:
        print("No samples to display")
        return
    
    # Choose random samples
    display_samples = random.sample(samples, min(num_samples, len(samples)))
    
    for i, sample in enumerate(display_samples):
        svg_content = sample['svg']
        description = sample['text']
        
        try:
            # Convert SVG to PNG for display
            png_data = cairosvg.svg2png(bytestring=svg_content.encode('utf-8'))
            img = Image.open(io.BytesIO(png_data))
            
            plt.figure(figsize=(5, 5))
            plt.imshow(img)
            plt.axis('off')
            plt.title(description)
            plt.show()
            
            print(f"Description: {description}")
            print(f"Category: {sample['category']}")
            print("SVG Content:")
            print(svg_content[:300] + '...' if len(svg_content) > 300 else svg_content)
            print("\n" + "-"*80 + "\n")
        except Exception as e:
            print(f"Error displaying sample {i}: {e}")


In [ ]:
# Display some processed samples
display_svg_samples(processed_samples, num_samples=3)

## Generate Better Text Descriptions (Optional)

For better training, we want more detailed and accurate descriptions. If available, we could use a vision model like BLIP-2 to generate captions for our SVGs:

In [ ]:
# This is optional and requires additional dependencies
def generate_captions_with_blip(samples):
    try:
        from transformers import BlipProcessor, BlipForConditionalGeneration
        import torch
        
        # Load BLIP model
        processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
        model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
        
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model = model.to(device)
        
        captioned_samples = []
        
        for sample in tqdm(samples, desc="Generating captions"):
            svg_content = sample['svg']
            
            try:
                # Convert SVG to PNG
                png_data = cairosvg.svg2png(bytestring=svg_content.encode('utf-8'))
                img = Image.open(io.BytesIO(png_data))
                
                # Generate caption
                inputs = processor(img, return_tensors="pt").to(device)
                out = model.generate(**inputs, max_length=30)
                caption = processor.decode(out[0], skip_special_tokens=True)
                
                # Create new sample with caption
                new_sample = sample.copy()
                new_sample['text'] = caption
                captioned_samples.append(new_sample)
            except Exception as e:
                print(f"Error generating caption: {e}")
                captioned_samples.append(sample)  # Keep the original sample
        
        return captioned_samples
    except ImportError:
        print("BLIP dependencies not installed, skipping caption generation")
        return samples

In [ ]:
# Generate better descriptions (optional)
# samples_with_captions = generate_captions_with_blip(processed_samples)
samples_with_captions = processed_samples  # Skip for now

## Split into Train/Val/Test Sets

In [ ]:
def split_dataset(samples, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-10, "Ratios must sum to 1"
    
    # Shuffle samples
    shuffled_samples = samples.copy()
    random.shuffle(shuffled_samples)
    
    # Calculate split indices
    n_samples = len(shuffled_samples)
    train_end = int(n_samples * train_ratio)
    val_end = train_end + int(n_samples * val_ratio)
    
    # Split data
    train_samples = shuffled_samples[:train_end]
    val_samples = shuffled_samples[train_end:val_end]
    test_samples = shuffled_samples[val_end:]
    
    return train_samples, val_samples, test_samples

In [ ]:
# Split dataset
train_samples, val_samples, test_samples = split_dataset(samples_with_captions)

print(f"Train samples: {len(train_samples)}")
print(f"Validation samples: {len(val_samples)}")
print(f"Test samples: {len(test_samples)}")

## Save Processed Data

In [ ]:
# Save processed data
save_json(train_samples, os.path.join(processed_data_dir, "train.json"))
save_json(val_samples, os.path.join(processed_data_dir, "val.json"))
save_json(test_samples, os.path.join(processed_data_dir, "test.json"))

print(f"Saved processed data to {processed_data_dir}")

## Check Token Lengths

Let's analyze the token lengths of our SVGs to make sure they fit within our model's context window:

In [ ]:
def analyze_token_lengths(samples, svg_processor):
    token_lengths = []
    command_counts = []
    
    for sample in tqdm(samples[:100], desc="Analyzing token lengths"):  # Limit to 100 samples for speed
        svg_content = sample['svg']
        
        # Get tokens
        tokens = svg_processor.svg_to_tokens(svg_content)
        token_lengths.append(len(tokens))
        
        # Get commands
        commands = svg_processor.svg_to_commands(svg_content)
        command_counts.append(len(commands))
    
    # Plot histograms
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    ax1.hist(token_lengths, bins=20)
    ax1.set_title('SVG Token Lengths')
    ax1.set_xlabel('Number of Tokens')
    ax1.set_ylabel('Count')
    
    ax2.hist(command_counts, bins=20)
    ax2.set_title('SVG Command Counts')
    ax2.set_xlabel('Number of Commands')
    ax2.set_ylabel('Count')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Token length statistics:")
    print(f"  Min: {min(token_lengths)}")
    print(f"  Max: {max(token_lengths)}")
    print(f"  Mean: {np.mean(token_lengths):.2f}")
    print(f"  Median: {np.median(token_lengths):.2f}")
    print(f"  95th percentile: {np.percentile(token_lengths, 95):.2f}")
    
    print(f"\nCommand count statistics:")
    print(f"  Min: {min(command_counts)}")
    print(f"  Max: {max(command_counts)}")
    print(f"  Mean: {np.mean(command_counts):.2f}")
    print(f"  Median: {np.median(command_counts):.2f}")
    print(f"  95th percentile: {np.percentile(command_counts, 95):.2f}")

In [ ]:
# Analyze token lengths
analyze_token_lengths(train_samples, svg_processor)

## Conclusion

We have successfully prepared a dataset for training the OmniSVG model. The data includes:

1. Simplified SVGs with standardized commands
2. Text descriptions for each SVG
3. Train/validation/test splits

The next steps would be to train the model using this data and then evaluate its performance on the test set.